In [11]:
from Bio.PDB import PDBParser, PDBIO, Select
import os
import requests
import pandas as pd

class ParsePDB:
    def __init__(self, pdb_name, chain_id=None, drug_id=None):
        self.pdb_name = pdb_name
        self.chain_id = chain_id
        self.drug_id = drug_id
        self.parser = PDBParser(QUIET=True)
        self.structure = self.parser.get_structure("struct", pdb_name)

    def accept_chain(self, chain):
        return chain.id == self.chain_id

    def accept_residue(self, residue):
        hetfield, _, _ = residue.get_id()
        return (hetfield == ' ' or residue.get_resname() == self.drug_id)

    def save_pdb(self, out_file):
        io = PDBIO()
        io.set_structure(self.structure)
        io.save(out_file)

    def keep_target(self, out_file):
        class ChainSelector(Select):
            def accept_chain(inner_self, chain):
                return self.accept_chain(chain)
        io = PDBIO()
        io.set_structure(self.structure)
        io.save(out_file, ChainSelector())

    def keep_drug(self, input_file, out_file):
        struct = self.parser.get_structure("filtered", input_file)
        class ResidueSelector(Select):
            def accept_residue(inner_self, residue):
                return self.accept_residue(residue)
        io = PDBIO()
        io.set_structure(struct)
        io.save(out_file, ResidueSelector())

    def keep_malaria(self, input_file, chain_id, out_file):
        struct = self.parser.get_structure("malaria", input_file)
        for model in struct:
            chains_to_remove = []
            for chain in model:
                if chain.id == chain_id:
                    chain.id = 'M'
                else:
                    chains_to_remove.append(chain)
            for ch in chains_to_remove:
                model.detach_child(ch.id)
        io = PDBIO()
        io.set_structure(struct)
        io.save(out_file)

    def rename_target_ligand(self, input_file, chain_id, out_file):
        struct = self.parser.get_structure("target", input_file)
        for model in struct:
            for chain in model:
                if chain.id == chain_id:
                    chain.id = 'T'
                for residue in chain:
                    hetfield, _, _ = residue.get_id()
                    if hetfield != ' ' and residue.get_resname() == self.drug_id:
                        residue.resname = 'D'
        io = PDBIO()
        io.set_structure(struct)
        io.save(out_file)

    @staticmethod
    def download_pdb(pdb_id, save_dir):
        """Download a PDB file from RCSB and return the local path."""
        pdb_id = pdb_id.strip()[:4].upper()
        url = f'https://files.rcsb.org/download/{pdb_id}.pdb'
        save_path = os.path.join(save_dir, f"{pdb_id}.pdb")

        if not os.path.exists(save_path):  # Avoid re-downloading
            try:
                response = requests.get(url, timeout=15)
                response.raise_for_status()
                with open(save_path, 'w') as f:
                    f.write(response.text)
                print(f"Downloaded: {pdb_id}")
            except Exception as e:
                print(f"Failed: {pdb_id} — {e}")
                return None
        else:
            print(f"Already exists: {pdb_id}")
        return save_path


In [12]:
import os
import pandas as pd

def run_preprocessing_pipeline(
    target_pdb_id, malaria_pdb_id,
    target_chain_id, malaria_chain_id,
    drug_resname,
    target_output_dir, malaria_output_dir,
    download_dir
):
    # Download PDBs first
    target_pdb_path = ParsePDB.download_pdb(target_pdb_id, download_dir)
    malaria_pdb_path = ParsePDB.download_pdb(malaria_pdb_id, download_dir)

    if not target_pdb_path or not malaria_pdb_path:
        print(f"Skipping {target_pdb_id} vs {malaria_pdb_id} due to download error.")
        return None, None

    # File basenames (without .pdb)
    target_basename = os.path.splitext(os.path.basename(target_pdb_path))[0]
    malaria_basename = os.path.splitext(os.path.basename(malaria_pdb_path))[0]

    # Sanitize ligand name (avoid invalid filename chars)
    safe_drug = str(drug_resname).replace("/", "-").replace(" ", "_")

    # Output paths (include chain ID and ligand ID in names)
    out_chain_only = os.path.join(target_output_dir, f"{target_basename}_{target_chain_id}_chain_only.pdb")
    out_ligand = os.path.join(target_output_dir, f"{target_basename}_{target_chain_id}_with_ligand.pdb")
    out_target = os.path.join(target_output_dir, f"{target_basename}_{target_chain_id}_{safe_drug}_renamed.pdb")
    out_malaria = os.path.join(malaria_output_dir, f"{malaria_basename}_{malaria_chain_id}_{safe_drug}_renamed.pdb")

    print(f"\nProcessing: {target_basename} (chain {target_chain_id}) vs {malaria_basename} (chain {malaria_chain_id})")
    print(" Target output:", out_target)
    print(" Malaria output:", out_malaria)

    # Processing
    parser = ParsePDB(target_pdb_path, target_chain_id, drug_resname)
    parser.keep_target(out_chain_only)
    parser.keep_drug(out_chain_only, out_ligand)
    parser.rename_target_ligand(out_ligand, target_chain_id, out_target)
    parser.keep_malaria(malaria_pdb_path, malaria_chain_id, out_malaria)

    return out_target, out_malaria


def main():
    # Directories
    download_dir = '/home/k_ensafitakaldani001_umb_edu/BLAST/pdb_downloads'
    malaria_output_dir = '/home/k_ensafitakaldani001_umb_edu/BLAST/malariapdb_renamed/'
    target_output_dir = '/home/k_ensafitakaldani001_umb_edu/BLAST/targetpdb_renamed/'

    os.makedirs(download_dir, exist_ok=True)
    os.makedirs(malaria_output_dir, exist_ok=True)
    os.makedirs(target_output_dir, exist_ok=True)

    # Load dataset
    csv_path = '/home/k_ensafitakaldani001_umb_edu/BLAST/NEW.csv'
    df = pd.read_csv(csv_path)

    # Ensure PDB IDs are 4-char format
    df['target'] = df['target'].apply(lambda x: str(x).strip().upper()[:4])
    df['malaria'] = df['malaria'].apply(lambda x: str(x).strip().upper()[:4])

    # Add columns for outputs
    df['target_processed'] = None
    df['malaria_processed'] = None

    # Run processing
    for idx, row in df.iterrows():
        try:
            out_target, out_malaria = run_preprocessing_pipeline(
                target_pdb_id=row['target'],
                malaria_pdb_id=row['malaria'],
                target_chain_id=row['T-cid'],
                malaria_chain_id=row['M-cid'],
                drug_resname=row['ligand_id'],
                target_output_dir=target_output_dir,
                malaria_output_dir=malaria_output_dir,
                download_dir=download_dir
            )
            if out_target and out_malaria:
                df.at[idx, 'target_processed'] = out_target
                df.at[idx, 'malaria_processed'] = out_malaria
        except Exception as e:
            print(f" Error processing row {idx}: {e}")

    # Save updated CSV with processed file paths
    out_csv = '/home/k_ensafitakaldani001_umb_edu/BLAST/NEW_with_processed.csv'
    df.to_csv(out_csv, index=False)
    print(f"\nSaved updated dataset to: {out_csv}")


if __name__ == "__main__":
    main()


Downloaded: 2BL9
Downloaded: 1J3J

Processing: 2BL9 (chain A) vs 1J3J (chain A)
 Target output: /home/k_ensafitakaldani001_umb_edu/BLAST/targetpdb_renamed/2BL9_A_CP6_renamed.pdb
 Malaria output: /home/k_ensafitakaldani001_umb_edu/BLAST/malariapdb_renamed/1J3J_A_CP6_renamed.pdb
Downloaded: 8TZB
Downloaded: 1V0B

Processing: 8TZB (chain A) vs 1V0B (chain A)
 Target output: /home/k_ensafitakaldani001_umb_edu/BLAST/targetpdb_renamed/8TZB_A_T3X_renamed.pdb
 Malaria output: /home/k_ensafitakaldani001_umb_edu/BLAST/malariapdb_renamed/1V0B_A_T3X_renamed.pdb
Downloaded: 8TYQ
Already exists: 1V0B

Processing: 8TYQ (chain A) vs 1V0B (chain A)
 Target output: /home/k_ensafitakaldani001_umb_edu/BLAST/targetpdb_renamed/8TYQ_A_T3X_renamed.pdb
 Malaria output: /home/k_ensafitakaldani001_umb_edu/BLAST/malariapdb_renamed/1V0B_A_T3X_renamed.pdb
Already exists: 8TYQ
Downloaded: 1OB3

Processing: 8TYQ (chain A) vs 1OB3 (chain A)
 Target output: /home/k_ensafitakaldani001_umb_edu/BLAST/targetpdb_renamed/8TY

/home/k_ensafitakaldani001_umb_edu/.local/lib/python3.13/site-packages/Bio/PDB/Entity.py:197: BiopythonWarning: The id `M` is already used for a sibling of this entity. Changing id from `L` to `M` might create access inconsistencies to children of the parent entity.
  warnings.warn(


Downloaded: 1QSG
Already exists: 3AM3

Processing: 1QSG (chain A) vs 3AM3 (chain A)
 Target output: /home/k_ensafitakaldani001_umb_edu/BLAST/targetpdb_renamed/1QSG_A_TCL_renamed.pdb
 Malaria output: /home/k_ensafitakaldani001_umb_edu/BLAST/malariapdb_renamed/3AM3_A_TCL_renamed.pdb
Already exists: 1QSG
Already exists: 3AM5

Processing: 1QSG (chain A) vs 3AM5 (chain A)
 Target output: /home/k_ensafitakaldani001_umb_edu/BLAST/targetpdb_renamed/1QSG_A_TCL_renamed.pdb
 Malaria output: /home/k_ensafitakaldani001_umb_edu/BLAST/malariapdb_renamed/3AM5_A_TCL_renamed.pdb
Already exists: 1QSG
Already exists: 1UH5

Processing: 1QSG (chain A) vs 1UH5 (chain A)
 Target output: /home/k_ensafitakaldani001_umb_edu/BLAST/targetpdb_renamed/1QSG_A_TCL_renamed.pdb
 Malaria output: /home/k_ensafitakaldani001_umb_edu/BLAST/malariapdb_renamed/1UH5_A_TCL_renamed.pdb
Already exists: 1QSG
Already exists: 2OL4

Processing: 1QSG (chain A) vs 2OL4 (chain A)
 Target output: /home/k_ensafitakaldani001_umb_edu/BLAST/t

/home/k_ensafitakaldani001_umb_edu/.local/lib/python3.13/site-packages/Bio/PDB/Entity.py:197: BiopythonWarning: The id `M` is already used for a sibling of this entity. Changing id from `L` to `M` might create access inconsistencies to children of the parent entity.
  warnings.warn(


Failed: 9NKG — 404 Client Error: Not Found for url: https://files.rcsb.org/download/9NKG.pdb
Already exists: 5FMG
Skipping 9NKG vs 5FMG due to download error.
Failed: 9NKG — 404 Client Error: Not Found for url: https://files.rcsb.org/download/9NKG.pdb
Already exists: 5FMG
Skipping 9NKG vs 5FMG due to download error.
Failed: 9NKG — 404 Client Error: Not Found for url: https://files.rcsb.org/download/9NKG.pdb
Already exists: 5FMG
Skipping 9NKG vs 5FMG due to download error.
Failed: 9NKG — 404 Client Error: Not Found for url: https://files.rcsb.org/download/9NKG.pdb
Already exists: 5FMG
Skipping 9NKG vs 5FMG due to download error.
Failed: 9NKG — 404 Client Error: Not Found for url: https://files.rcsb.org/download/9NKG.pdb
Already exists: 5FMG
Skipping 9NKG vs 5FMG due to download error.
Failed: 9NKG — 404 Client Error: Not Found for url: https://files.rcsb.org/download/9NKG.pdb
Already exists: 5FMG
Skipping 9NKG vs 5FMG due to download error.
Failed: 9NKG — 404 Client Error: Not Found for

In [1]:
#its an example to see if we are doing it right

In [13]:
from Bio.PDB import PDBParser, PPBuilder

def parse_chains_and_ligands_with_atom_sequences(pdb_file):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('pdb_structure', pdb_file)
    ppb = PPBuilder()

    results = {}

    for model in structure:
        for chain in model:
            chain_id = chain.id

            # Extract protein sequence
            seq = ''
            for pp in ppb.build_peptides(chain):
                seq += str(pp.get_sequence())

            # Extract ligands connected to this chain with their "atom sequence"
            ligands = []
            for residue in chain:
                hetfield, resseq, icode = residue.get_id()
                if hetfield != ' ' and residue.get_resname() != 'HOH':
                    atom_sequence = []
                    for atom in residue:
                        atom_sequence.append({
                            'atom_name': atom.get_name(),
                            'element': atom.element
                        })
                    ligands.append({
                        'residue_name': residue.get_resname(),
                        'resseq': resseq,
                        'insertion_code': icode,
                        'atom_sequence': atom_sequence
                    })

            results[chain_id] = {
                'protein_sequence': seq,
                'ligands': ligands
            }

    return results


# Example usage

!wget -qnc https://files.rcsb.org/download/3am3.pdb

pdb_file = '/home/k_ensafitakaldani001_umb_edu/BLAST/targetpdb_renamed/2B35_A_TCL_renamed.pdb'  
data = parse_chains_and_ligands_with_atom_sequences(pdb_file)

# Print results
for chain_id, content in data.items():
    print(f"=== Chain {chain_id} ===")
    print("Protein Sequence:")
    print(content['protein_sequence'])
    print("Ligands:")
    if content['ligands']:
        for lig in content['ligands']:
            print(f"  Ligand {lig['residue_name']} {lig['resseq']}{lig['insertion_code']}:")
            print("    Atom Sequence:")
            atom_seq_str = ' - '.join([f"{a['atom_name']}({a['element']})" for a in lig['atom_sequence']])
            print(f"    {atom_seq_str}")
    else:
        print("  None")
    print()


=== Chain T ===
Protein Sequence:
TGLLDGKRILVSGIITDSSIAFHIARVAQEQGAQLVLTGFDRLRLIQRITDRLPAKAPLLELDVQNEEHLASLAGRVTEAIGAGNKLDGVVHSIGFMPQTGMGINPFFDAPYADVSKGIHISAYSYASMAKALLPIMNPGGSIVGMDFDPSRAMPAYNWMTVAKSALESVNRFVAREAGKYGVRSNLVAAGPIRTGAQIQLLEEGWDQRAPIGWNMKDATPVAKTVCALLSDWLPATTGDIIYADGGAHTQLL
Ligands:
  Ligand D 300 :
    Atom Sequence:
    C1(C) - C2(C) - C6(C) - C5(C) - C4(C) - C3(C) - C11(C) - C10(C) - C9(C) - C8(C) - C12(C) - C13(C) - O7(O) - CL14(CL) - CL15(CL) - CL16(CL) - O17(O)



In [14]:
from Bio.PDB import PDBParser, NeighborSearch, Selection, PPBuilder

def find_ligand_binding_sites(pdb_file, distance_cutoff=5.0):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('pdb_structure', pdb_file)

    model = next(structure.get_models())  # get first model

    # Build list of all protein atoms for NeighborSearch
    atoms = Selection.unfold_entities(model, 'A')  # 'A' for atoms
    ns = NeighborSearch(atoms)

    results = []

    for chain in model:
        chain_id = chain.id

        # Extract protein sequence
        ppb = PPBuilder()
        seq = ''
        for pp in ppb.build_peptides(chain):
            seq += str(pp.get_sequence())

        # Find ligands in this chain
        for residue in chain:
            hetfield, resseq, icode = residue.get_id()
            if hetfield != ' ' and residue.get_resname() != 'HOH':
                ligand_atoms = list(residue.get_atoms())
                ligand_name = residue.get_resname()

                # Find nearby residues (binding site)
                binding_residues = set()
                for atom in ligand_atoms:
                    neighbors = ns.search(atom.get_coord(), distance_cutoff, level='R')
                    for neighbor in neighbors:
                        if neighbor.get_parent().id == chain_id:
                            # Only include protein residues, skip heteroatoms
                            n_hetfield = neighbor.get_id()[0]
                            if n_hetfield == ' ':
                                res_id = neighbor.get_id()[1]
                                res_name = neighbor.get_resname()
                                binding_residues.add( (res_id, res_name) )

                results.append({
                    'chain_id': chain_id,
                    'ligand_name': ligand_name,
                    'ligand_resseq': resseq,
                    'binding_site_residues': sorted(binding_residues)
                })

    return results

# Example usage
pdb_file = '/home/k_ensafitakaldani001_umb_edu/BLAST/targetpdb_renamed/2B35_A_TCL_renamed.pdb'  
binding_sites = find_ligand_binding_sites(pdb_file)

# Print results
for entry in binding_sites:
    print(f"Chain {entry['chain_id']} - Ligand {entry['ligand_name']} {entry['ligand_resseq']}")
    print("Binding site residues:")
    for res in entry['binding_site_residues']:
        print(f"  Residue {res[1]} {res[0]}")
    print()


Chain T - Ligand D 300
Binding site residues:
  Residue GLY 96
  Residue PHE 97
  Residue MET 98
  Residue MET 103
  Residue PHE 149
  Residue TYR 158
  Residue MET 161
  Residue LYS 165
  Residue PRO 193
  Residue ILE 194

